In [3]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import glob
from pathlib import Path
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader,TextLoader
from langchain_chroma import Chroma
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_openai import ChatOpenAI

In [4]:
load_dotenv(override=True)
embedding_model=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
model="llama3.2:1b "

In [11]:
KNOWLEDGE_BASE=Path("knowledge-base")
DB_NAME=Path("vector_db")

In [13]:
folders=glob.glob(str(KNOWLEDGE_BASE/"*"))
documents=[]
for folder in folders:
    doc_type=os.path.basename(folder)
    loader=DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={"encoding":"utf-8"})
    folder_docs=loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

In [14]:
print(documents)

[Document(metadata={'source': 'knowledge-base\\HR-policies\\employee_benefits.md', 'doc_type': 'HR-policies'}, page_content='# Employee Benefits and Compensation Policy\n\n## 1. Health Insurance\n\n- Coverage for employees and dependents\n\n---\n\n## 2. Provident Fund (PF)\n\n- Company contributes as per regulations\n\n---\n\n## 3. Gratuity\n\n- Eligible after 5 years of continuous service\n\n---\n\n## 4. Performance Bonus\n\n- Annual bonus based on:\n  - Individual performance\n  - Company performance\n\n---\n\n## 5. Flexible Benefits\n\nEmployees can choose:\n- Meal vouchers\n- Internet reimbursement\n- Wellness programs\n\n---\n\n## 6. Learning and Development\n\n- Up to INR 50,000 per year reimbursement\n- Covers:\n  - Courses\n  - Certifications\n  - Workshops\n\n---\n\n## 7. Parental Leave\n\n- Maternity Leave: 26 weeks\n- Paternity Leave: 10 days'), Document(metadata={'source': 'knowledge-base\\HR-policies\\leave_policy.md', 'doc_type': 'HR-policies'}, page_content='# Comprehens

In [17]:
headers_to_split_on = [
    ("#", "h1"),
    ("##", "h2"),
]

text_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

chunks = []

for doc in documents:
    split_docs = text_splitter.split_text(doc.page_content)

    for chunk in split_docs:
        # preserve original metadata
        chunk.metadata.update(doc.metadata)
        chunks.append(chunk)
print(chunks)

[Document(metadata={'h1': 'Employee Benefits and Compensation Policy', 'h2': '1. Health Insurance', 'source': 'knowledge-base\\HR-policies\\employee_benefits.md', 'doc_type': 'HR-policies'}, page_content='- Coverage for employees and dependents  \n---'), Document(metadata={'h1': 'Employee Benefits and Compensation Policy', 'h2': '2. Provident Fund (PF)', 'source': 'knowledge-base\\HR-policies\\employee_benefits.md', 'doc_type': 'HR-policies'}, page_content='- Company contributes as per regulations  \n---'), Document(metadata={'h1': 'Employee Benefits and Compensation Policy', 'h2': '3. Gratuity', 'source': 'knowledge-base\\HR-policies\\employee_benefits.md', 'doc_type': 'HR-policies'}, page_content='- Eligible after 5 years of continuous service  \n---'), Document(metadata={'h1': 'Employee Benefits and Compensation Policy', 'h2': '4. Performance Bonus', 'source': 'knowledge-base\\HR-policies\\employee_benefits.md', 'doc_type': 'HR-policies'}, page_content='- Annual bonus based on:\n- I